# DAT535 project - results

Shows the output of the last pipeline run (`python run_pipeline.py all`).
Data: BTS On-Time Performance 2025 (7M flights) + OpenFlights airports.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
DATA = Path(os.getenv("DAT535_OUTPUT_DIR") or ROOT / "data")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import ingest
import run_pipeline
from publish import current_run

spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("DAT535-Results")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
pd.set_option("display.width", 200)


def read_json(path):
    return json.loads(Path(path).read_text())

26/11/11 08:35:26 WARN Utils: Your hostname, denzo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/11 08:35:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/11 08:35:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Bronze

In [2]:
receipt = read_json(DATA / "bronze" / "manifests" / "delivery_2025.json")
pd.DataFrame([{
    "year": receipt["year"],
    "status": receipt["status"],
    "months": f"{receipt['received_count']}/{receipt['expected_count']}",
    "missing": receipt["missing_months"],
    "records": f"{receipt['source_records']:,}",
    "schema versions": len(receipt["schema_versions"]),
}])

,year,status,months,missing,records,schema versions
0,2025,COMPLETE,12/12,[],"7,001,619",1


In [3]:
manifests = [read_json(p) for p in sorted((DATA / "bronze" / "manifests").glob("flights_2025_*.json"))]
pd.DataFrame([{
    "month": m["month"],
    "rows": f"{m['row_count']:,}",
    "MB": round(m["source_size_bytes"] / 1e6, 1),
    "sha256": m["checksum_sha256"][:12],
    "schema": m.get("schema_hash", m.get("schema_fingerprint")),
} for m in manifests])

,month,rows,MB,sha256,schema
0,1,"539,747",243.2,d7c7d59452ca,5965ede3dc889115
1,2,"504,884",228.2,fd6b7aa423cf,5965ede3dc889115
2,3,"600,872",271.6,ba031a484a67,5965ede3dc889115
3,4,"583,950",264.1,1124d8e68633,5965ede3dc889115
4,5,"605,648",274.3,644c6d36949a,5965ede3dc889115
5,6,"611,575",277.4,a5b0b743e01e,5965ede3dc889115
6,7,"631,428",286.2,8e1889b42eb9,5965ede3dc889115
7,8,"602,378",272.7,8300437f9b54,5965ede3dc889115
8,9,"562,439",254.1,6b189acc187f,5965ede3dc889115
9,10,"605,844",274.7,1162446fa628,5965ede3dc889115


In [4]:
bronze = spark.read.parquet(str(DATA / "bronze" / "flights"))
print(f"{bronze.count():,} rows, {len(bronze.columns)} columns (all strings)")
bronze.select(*ingest.REQUIRED_COLUMNS, "_source_file").printSchema()

7,001,619 rows, 114 columns (all strings)
root
 |-- FlightDate: string (nullable = true)
 |-- Reporting_Airline: string (nullable = true)
 |-- Flight_Number_Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- CRSDepTime: string (nullable = true)
 |-- DepTime: string (nullable = true)
 |-- DepDelay: string (nullable = true)
 |-- CRSArrTime: string (nullable = true)
 |-- ArrTime: string (nullable = true)
 |-- ArrDelay: string (nullable = true)
 |-- Cancelled: string (nullable = true)
 |-- Diverted: string (nullable = true)
 |-- CRSElapsedTime: string (nullable = true)
 |-- Distance: string (nullable = true)
 |-- _source_file: string (nullable = true)



In [5]:
bronze.select("FlightDate", "Reporting_Airline", "Origin", "Dest", "CRSDepTime",
              "DepDelay", "ArrDelay", "Cancelled", "_source_file").show(5, truncate=False)

+----------+-----------------+------+----+----------+--------+--------+---------+-----------------------------------------------------------------------+
|FlightDate|Reporting_Airline|Origin|Dest|CRSDepTime|DepDelay|ArrDelay|Cancelled|_source_file                                                           |
+----------+-----------------+------+----+----------+--------+--------+---------+-----------------------------------------------------------------------+
|2025-07-17|UA               |SAT   |DEN |1343      |-2.00   |-22.00  |0.00     |On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_7.csv|
|2025-07-17|UA               |BOS   |ORD |1040      |2.00    |-6.00   |0.00     |On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_7.csv|
|2025-07-17|UA               |ORD   |DFW |0845      |30.00   |13.00   |0.00     |On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_7.csv|
|2025-07-17|UA               |ORF   |ORD |0620      |1.00    |10.00   |0.00 

## Silver

In [6]:
silver_run = current_run(DATA / "silver")
report = read_json(silver_run / "quality_report.json")
print("Silver run:", silver_run.name)

pd.DataFrame([
    ("Bronze records", report["source_records"]),
    ("duplicates removed", report["duplicates_removed"]),
    ("rejected", report["rejected_records"]),
    ("accepted", report["accepted_records"]),
    ("unmatched airports", report["unmatched_origin_records"] + report["unmatched_destination_records"]),
    ("missing UTC", report["missing_utc"]),
    ("arrival time mismatches", report["scheduled_arrival_mismatches"]),
], columns=["", "count"]).assign(count=lambda d: d["count"].map("{:,}".format))

Silver run: f08bbe87-e56d-4bfe-b977-102b406660c1


,,count
0,Bronze records,"7,001,619"
1,duplicates removed,0
2,rejected,1
3,accepted,"7,001,618"
4,unmatched airports,0
5,missing UTC,0
6,arrival time mismatches,213


In [7]:
gate = report["quality_gate"]
print("Quality gate:", gate["status"])
pd.DataFrame([
    {"check": key, "limit": limit, "value": round(gate["measures"][key.split("_", 1)[1]], 6)}
    for key, limit in gate["limits"].items()
])

Quality gate: PASS


,check,limit,value
0,max_rejected_pct,1.0,0.000014
1,max_unmatched_airport_pct,0.0,0.000000
2,max_time_mismatch_pct,0.1,0.002079
3,max_identity_conflicts,0.0,0.000000
4,max_missing_utc,0.0,0.000000
5,min_time_check_coverage_pct,99.0,99.999900


Missing / unparseable values per column, Bronze vs Silver:

In [8]:
pd.DataFrame.from_dict(report["before_after"], orient="index")

,silver_column,bronze_missing,bronze_unparseable,silver_missing
FlightDate,flight_date,0,0,0
Reporting_Airline,airline,0,0,0
Flight_Number_Reporting_Airline,flight_number,0,0,0
Origin,origin,0,0,0
Dest,destination,0,0,0
CRSDepTime,scheduled_departure,0,0,0
DepTime,actual_departure,98170,0,98170
DepDelay,departure_delay,98591,0,98591
CRSArrTime,scheduled_arrival,0,0,0
ArrTime,actual_arrival,104608,0,104607


In [9]:
print("Reject reasons:", report["rejection_reasons"])
rejected = spark.read.parquet(str(silver_run / "rejected"))
rejected.select("flight_date", "airline", "flight_number", "origin", "destination",
                "flight_status", "arrival_delay", "reject_reason").show(truncate=False)

Reject reasons: {'missing_arrival_delay': 1}
+-----------+-------+-------------+------+-----------+-------------+-------------+-----------------------+
|flight_date|airline|flight_number|origin|destination|flight_status|arrival_delay|reject_reason          |
+-----------+-------+-------------+------+-----------+-------------+-------------+-----------------------+
|2025-09-06 |YX     |5859         |JFK   |MVY        |completed    |NULL         |[missing_arrival_delay]|
+-----------+-------+-------------+------+-----------+-------------+-------------+-----------------------+



In [10]:
flights = spark.read.parquet(str(silver_run / "flights"))
print(f"{flights.count():,} rows, {len(flights.columns)} columns, partitioned by year/month")
flights.printSchema()

7,001,618 rows, 45 columns, partitioned by year/month
root
 |-- destination: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- airline: string (nullable = true)
 |-- flight_number: integer (nullable = true)
 |-- scheduled_departure: integer (nullable = true)
 |-- departure_delay: double (nullable = true)
 |-- arrival_delay: double (nullable = true)
 |-- cancelled: integer (nullable = true)
 |-- diverted: integer (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- scheduled_arrival: integer (nullable = true)
 |-- scheduled_elapsed_min: double (nullable = true)
 |-- actual_departure: integer (nullable = true)
 |-- actual_arrival: integer (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingestion_time: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- flight_status: string (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- departure_hour: long (nullable = true)

In [11]:
flights.select("flight_date", "airline", "origin", "destination", "scheduled_departure",
               "scheduled_departure_utc", "actual_arrival_utc", "arrival_delay",
               "flight_status").orderBy("flight_date", "scheduled_departure").show(10)

+-----------+-------+------+-----------+-------------------+-----------------------+-------------------+-------------+-------------+
|flight_date|airline|origin|destination|scheduled_departure|scheduled_departure_utc| actual_arrival_utc|arrival_delay|flight_status|
+-----------+-------+------+-----------+-------------------+-----------------------+-------------------+-------------+-------------+
| 2025-01-01|     DL|   SFO|        MSP|                 10|    2025-01-01 08:10:00|2025-01-01 11:26:00|        -33.0|    completed|
| 2025-01-01|     AA|   SFO|        CLT|                 17|    2025-01-01 08:17:00|2025-01-01 12:38:00|        -40.0|    completed|
| 2025-01-01|     AA|   SMF|        ORD|                 25|    2025-01-01 08:25:00|2025-01-01 12:18:00|        -16.0|    completed|
| 2025-01-01|     AS|   FAI|        SEA|                 25|    2025-01-01 09:25:00|2025-01-01 12:34:00|        -29.0|    completed|
| 2025-01-01|     AA|   LAX|        ORD|                 27|    2025-

In [12]:
history = [json.loads(line) for line in (DATA / "silver" / "quality_history.jsonl").read_text().splitlines()]
pd.DataFrame(history)[["created_utc", "gate_status", "source_records", "accepted_records",
                       "rejected_records", "time_mismatch_pct"]]

,created_utc,gate_status,source_records,accepted_records,rejected_records,time_mismatch_pct
0,2026-11-11T07:34:53.284692+00:00,PASS,7001619,7001618,1,0.002079


## Gold

In [13]:
gold_run = current_run(DATA / "gold")
manifest = read_json(gold_run / "manifest.json")
print("Gold run:", gold_run.name, "- built from Silver run", manifest["silver_run_id"])
pd.DataFrame([{"table": name, "rows": info["aggregate_rows"]} for name, info in manifest["datasets"].items()])

Gold run: 0b1d9337-6a31-4555-9bf1-172c1286eedb - built from Silver run f08bbe87-e56d-4bfe-b977-102b406660c1


,table,rows
0,airline_performance,168
1,airport_performance,4073
2,daily_trends,365


Airline ranking, July 2025:

In [14]:
airlines = spark.read.parquet(str(gold_run / "airline_performance"))
airlines.filter("year = 2025 AND month = 7").orderBy("punctuality_rank").select(
    "punctuality_rank", "airline", "total_flights", "arrival_delay_rate_pct",
    "cancellation_rate_pct").show(20)

+----------------+-------+-------------+----------------------+---------------------+
|punctuality_rank|airline|total_flights|arrival_delay_rate_pct|cancellation_rate_pct|
+----------------+-------+-------------+----------------------+---------------------+
|               1|     HA|         7066|                 20.43|                 0.72|
|               2|     OO|        75877|                 24.66|                 1.35|
|               3|     DL|        94066|                 24.84|                  1.4|
|               4|     NK|        17307|                 24.89|                  1.9|
|               5|     MQ|        27489|                 26.36|                 2.32|
|               6|     YX|        30396|                 27.26|                 8.97|
|               7|     WN|       125677|                 28.93|                 1.28|
|               8|     G4|        13978|                  30.1|                 0.44|
|               9|     UA|        69788|              

Delta (DL) month by month:

In [15]:
airlines.filter("airline = 'DL'").orderBy("month").select(
    "month", "arrival_delay_rate_pct", "previous_month_delay_rate_pct",
    "delay_rate_change_pts", "cancelled_flights", "cancellations_ytd").show(12)

+-----+----------------------+-----------------------------+---------------------+-----------------+-----------------+
|month|arrival_delay_rate_pct|previous_month_delay_rate_pct|delay_rate_change_pts|cancelled_flights|cancellations_ytd|
+-----+----------------------+-----------------------------+---------------------+-----------------+-----------------+
|    1|                 19.32|                         NULL|                 NULL|             2137|             2137|
|    2|                 19.97|                        19.32|                 0.65|              164|             2301|
|    3|                 18.42|                        19.97|                -1.55|              416|             2717|
|    4|                 17.77|                        18.42|                -0.65|              430|             3147|
|    5|                 22.01|                        17.77|                 4.24|              195|             3342|
|    6|                 26.17|                  

Worst departure airports (min. 20 000 flights):

In [16]:
airports = spark.read.parquet(str(gold_run / "airport_performance"))
(airports.groupBy("origin", "origin_airport_name")
    .agg(F.sum("total_flights").alias("flights"),
         F.sum("arrival_delayed_15_flights").alias("late"),
         F.sum("completed_flights").alias("completed"))
    .filter("flights >= 20000")
    .withColumn("late_pct", F.round(100 * F.col("late") / F.col("completed"), 2))
    .orderBy(F.desc("late_pct"))
    .show(10, truncate=False))

+------+-----------------------------------------+-------+-----+---------+--------+
|origin|origin_airport_name                      |flights|late |completed|late_pct|
+------+-----------------------------------------+-------+-----+---------+--------+
|DFW   |Dallas Fort Worth International Airport  |315854 |87789|306185   |28.67   |
|DCA   |Ronald Reagan Washington National Airport|142506 |37089|136684   |27.13   |
|ORD   |Chicago O'Hare International Airport     |327028 |84927|320418   |26.51   |
|LGA   |La Guardia Airport                       |135725 |34688|131177   |26.44   |
|PHL   |Philadelphia International Airport       |97519  |25102|95211    |26.36   |
|MIA   |Miami International Airport              |111907 |28671|110113   |26.04   |
|DEN   |Denver International Airport             |317686 |80758|314281   |25.7    |
|EWR   |Newark Liberty International Airport     |120286 |29538|116940   |25.26   |
|PBI   |Palm Beach International Airport         |30785  |7574 |30181    |25

Worst 7-day periods:

In [17]:
daily = spark.read.parquet(str(gold_run / "daily_trends"))
daily.orderBy(F.desc("arrival_delay_rate_7d_pct")).select(
    "flight_date", "total_flights", "arrival_delay_rate_pct",
    "arrival_delay_rate_7d_pct", "cancellation_rate_7d_pct").show(10)

+-----------+-------------+----------------------+-------------------------+------------------------+
|flight_date|total_flights|arrival_delay_rate_pct|arrival_delay_rate_7d_pct|cancellation_rate_7d_pct|
+-----------+-------------+----------------------+-------------------------+------------------------+
| 2025-02-18|        17724|                 26.91|                    35.27|                    2.89|
| 2025-02-19|        17539|                 28.85|                    34.71|                    3.04|
| 2025-02-17|        19815|                 38.52|                    34.48|                    2.82|
| 2025-07-14|        21445|                 37.85|                    33.96|                    4.58|
| 2025-07-16|        19898|                  30.9|                    33.74|                    4.36|
| 2025-07-15|        19703|                  28.0|                    33.71|                    4.37|
| 2025-12-05|        19238|                 28.97|                    33.27|      

Gold totals vs Silver:

In [18]:
check = pd.DataFrame({name: info["metrics"] for name, info in manifest["datasets"].items()})
check["silver"] = pd.Series(manifest["source_metrics"])
print("All equal:", bool(check.drop(columns="silver").eq(check["silver"], axis=0).all().all()))
check

All equal: True


,airline_performance,airport_performance,daily_trends,silver
total_flights,7001618,7001618,7001618,7001618
completed_flights,6879484,6879484,6879484,6879484
cancelled_flights,102876,102876,102876,102876
diverted_flights,19258,19258,19258,19258
arrival_delayed_15_flights,1534638,1534638,1534638,1534638


## Performance

In [19]:
logs = sorted(run_pipeline.log_dir().glob("pipeline_*.json"))[-5:]
pd.DataFrame([
    {"run": p.stem[9:24],
     **{stage: round(info.get("elapsed_time", 0), 1) for stage, info in read_json(p)["stages"].items()},
     "total": round(read_json(p)["total_time"], 1)}
    for p in logs
])

,run,bronze,silver,gold,total
0,20261111T073457,28.3,60.1,3.5,91.9


In [20]:
perf_dir = DATA / "logs" / "performance"
if not perf_dir.exists():
    perf_dir = ROOT / "logs" / "performance"
print((perf_dir / "query_summary.txt").read_text())

DAT535 - Spark query optimization
Data: BTS On-Time Performance 2025, Silver run f08bbe87-e56d-4bfe-b977-102b406660c1
Flights: 7,001,618
Times: median of 5 runs after 1 warm-up

1. BROADCAST JOIN (airport names onto flights, origin + destination)
   with broadcast:    BroadcastHashJoin LeftOuter BuildRight (19)  0.64 s
   without broadcast: SortMergeJoin LeftOuter (25)  2.02 s
   same result: 7,001,618 rows in both

2. PARTITION PRUNING (filter month = 7)
   PartitionFilters: [isnotnull(month#44), (month#44 = 7)]

3. COLUMN PRUNING (select 2 of 45 Silver columns)
   ReadSchema: struct<airline:string,arrival_delay:double>

4. FILTER PUSHDOWN (filter airline = 'HA', not a partition column)
   PushedFilters: [IsNotNull(airline), EqualTo(airline,HA)]
   all airlines 0.04 s, HA only 0.08 s
   no speed-up: Silver is not sorted by airline, so no row groups can be skipped



## Summary

In [21]:
for line in run_pipeline.summary_lines():
    print(line)

spark.stop()

  Bronze 2025: COMPLETE, 12 months, 7,001,619 records
  Silver last run (PASS): 7,001,619 in -> 7,001,618 accepted, 1 rejected, 0 duplicates removed
  Gold airline_performance: 168 rows
  Gold airport_performance: 4,073 rows
  Gold daily_trends: 365 rows
  Gold run 0b1d9337-6a31-4555-9bf1-172c1286eedb built from Silver f08bbe87-e56d-4bfe-b977-102b406660c1
